# Proyecto 2 – Space Invaders (Rainbow DQN) en Google Colab / Kaggle

**Antes de empezar:** `Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)`.

Este cuaderno:
1. Monta Google Drive (los checkpoints se guardan ahí para sobrevivir desconexiones).
2. Clona el repositorio del proyecto e instala dependencias.
3. Entrena (`train.py`), reanuda si se cortó (`--resume`), evalúa y genera el video (`evaluate.py`).

> En Colab gratuito la sesión se corta a las ~12 h (a veces antes). Como `train.py` guarda `last.pt`
> cada `--save-every` pasos, basta volver a ejecutar la celda de **reanudar**.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/proyecto2_spaceinvaders'   # carpeta persistente en Drive
RUNS_DIR = f'{DRIVE_DIR}/runs'
os.makedirs(RUNS_DIR, exist_ok=True)

# Opción A: clonar el repo de GitHub (recomendado). Cambia la URL por la de tu repositorio.
REPO_URL = 'https://github.com/Brariv/proyecto2-spaceinvaders.git'
if not os.path.exists('/content/proyecto2'):
    !git clone -q {REPO_URL} /content/proyecto2
# Opción B: si subiste la carpeta del proyecto a Drive, usa:
#   !cp -r "{DRIVE_DIR}/codigo" /content/proyecto2
%cd /content/proyecto2
!git pull -q

In [ ]:
%pip install -q "gymnasium>=1.1" "ale-py>=0.11" imageio imageio-ffmpeg tabulate
import torch, gymnasium, ale_py, os
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
print('gymnasium', gymnasium.__version__, '| ale_py', ale_py.__version__, '| CPUs', os.cpu_count())
!python tests/test_replay.py

## Entrenar

Presets disponibles (ver `si_rl/presets.py`): `dqn`, `ddqn`, `dueling`, `per_nstep`, `rainbow`, `rainbow_c51`, `rainbow_fast`, `rainbow_fast_small`.

Con 2 vCPUs de Colab, `rainbow_fast` corre a ~600-900 pasos/s (10M pasos ≈ 3-5 h).

In [ ]:
RUN_NAME = 'it6_rainbow_fast'
!python train.py --preset rainbow_fast --run-name {RUN_NAME} --runs-dir "{RUNS_DIR}" \
    --total-steps 10_000_000 --save-every 250_000 --eval-every 250_000 --eval-episodes 5

## Reanudar (si se cortó la sesión)
Vuelve a ejecutar las dos primeras celdas (Drive + instalación) y luego esta.

In [ ]:
RUN_NAME = 'it6_rainbow_fast'
!python train.py --resume "{RUNS_DIR}/{RUN_NAME}/last.pt" --run-name {RUN_NAME} --runs-dir "{RUNS_DIR}" \
    --total-steps 10_000_000

## Curvas (TensorBoard o gráficas estáticas)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir "{RUNS_DIR}" 

In [ ]:
!python plot_results.py "{RUNS_DIR}"/it* --out "{DRIVE_DIR}/figs"
from IPython.display import Image, display
display(Image(f'{DRIVE_DIR}/figs/curvas_entrenamiento.png'))
display(Image(f'{DRIVE_DIR}/figs/curvas_evaluacion.png'))

## Evaluar (5 episodios greedy) y generar el video

In [ ]:
RUN_NAME = 'it6_rainbow_fast'
# 1) elegir el mejor checkpoint de la corrida (10 episodios por checkpoint)
!python evaluate.py --checkpoints-dir "{RUNS_DIR}/{RUN_NAME}" --episodes 10 --json "{DRIVE_DIR}/ranking_ckpts.json"


In [ ]:
CHECKPOINT = f'{RUNS_DIR}/{RUN_NAME}/best.pt'      # o el mejor según el ranking anterior
VIDEO = f'{DRIVE_DIR}/agente_final.mp4'
!python evaluate.py --checkpoint "{CHECKPOINT}" --episodes 5 --video "{VIDEO}" --json "{DRIVE_DIR}/resultado_final.json"

# copiar los pesos finales al repo para entregarlos
!mkdir -p modelo_final && cp "{CHECKPOINT}" modelo_final/best.pt

from IPython.display import HTML
from base64 import b64encode
mp4 = open(VIDEO, 'rb').read()
HTML(f'<video width=400 controls autoplay loop><source src="data:video/mp4;base64,{b64encode(mp4).decode()}" type="video/mp4"></video>')

## Notas para Kaggle
* `Settings → Accelerator → GPU T4 x2 / P100` y `Internet → On`. Kaggle da ~30 h de GPU por semana en sesiones de hasta 12 h (4 vCPUs: usa `--num-envs 32` sin problema).
* No hay Drive: usa `--runs-dir /kaggle/working/runs` y al final descarga `best.pt` / `last.pt` (o guárdalos como *Dataset* de Kaggle para reanudar en otra sesión con `--resume`).